In [1]:
#pip install pandas
#pip install fasttext

In [9]:
import pandas as pd 
import fasttext
import re 

modelo_detector = "/home/pc/Downloads/Dados MDA/teste/script_detector_idioma/lid.176.bin" #modelo salvo no arquivo script_detector_idioma

def detectar_idioma(modelo, texto):
    if not isinstance(texto, str) or not texto.strip():
        return None

    try:
        predicao = modelo.predict(texto.replace("\n", " "))
        idioma = predicao[0][0].replace("__label__", "")
        return idioma
    except Exception:
        return None
        

In [10]:
def limpar_texto(texto):
    if pd.isna(texto):
        return ""

    texto = re.sub(r'http\S+|www\S+|https\S+', '', texto)
    
    texto = re.sub(r'\b(k+|rs+|ha+|he+|hi+|kk+|ka+){2,}\b', '', texto, flags=re.IGNORECASE)

    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto

In [11]:
def processar_csv(caminho_entrada, caminho_saida):
    print("Carregando modelo")
    modelo = fasttext.load_model(modelo_detector)

    df = pd.read_csv(caminho_entrada)

    if "post_description" not in df.columns:
        raise ValueError ("Coluna 'post_description' não encontrada.")

    if "post_validation_status" not in df.columns:
        df["post_validation_status"]=""

    for i, texto in enumerate(df["post_description"]):
        idioma = detectar_idioma(modelo, texto)

        if idioma != "pt":
            df.at[i, "post_validation_status"] = "INVALID"

        else:
             df.at[i, "post_validation_status"] = "VALID"
            

    df.to_csv(caminho_saida, index=False)

    print ("\nProcessamento concluído.")
    print ("Arquivo salvo em:", caminho_saida)

In [12]:
def main():
    caminho_entrada = input("Digite o caminho do csv de entrada: ").strip()
    caminho_saida = input("Digite o caminho do CSV de saída: ").strip()

    processar_csv(caminho_entrada, caminho_saida)



In [15]:
if __name__== "__main__":
    main()

Digite o caminho do csv de entrada:  /home/pc/Downloads/Dados MDA/16-22_03_2026/x_2026-03-16_2026-03-22_antes.csv
Digite o caminho do CSV de saída:  /home/pc/Downloads/Dados MDA/16-22_03_2026/x_2026-03-16_2026-03-22.csv


Carregando modelo


/tmp/ipykernel_145466/3746311658.py:17: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'INVALID' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.at[i, "post_validation_status"] = "INVALID"



Processamento concluído.
Arquivo salvo em: /home/pc/Downloads/Dados MDA/16-22_03_2026/x_2026-03-16_2026-03-22.csv
